In [7]:
import pandas as pd
from utils.notebook_helpers import get_postgres_engine

engine, pg_conf = get_postgres_engine("../configs/database.yaml")
print("Connected to PostgreSQL:", pg_conf["host"], pg_conf["db"])

Connected to PostgreSQL: localhost storage_analytics


In [8]:
# Dashboard mart loads
device_overview = pd.read_sql("SELECT * FROM mart_tableau_device_overview", engine)
anomaly_timeline = pd.read_sql("SELECT * FROM mart_tableau_anomaly_timeline", engine)
root_cause_summary = pd.read_sql("SELECT * FROM mart_tableau_root_cause_summary", engine)
grafana_health = pd.read_sql("SELECT * FROM v_grafana_device_health", engine)

datasets = {
    "device_overview": device_overview,
    "anomaly_timeline": anomaly_timeline,
    "root_cause_summary": root_cause_summary,
    "grafana_health": grafana_health,
}

for name, df in datasets.items():
    print(f"{name}: {len(df):,} rows, {df.shape[1]} columns")

device_overview: 5 rows, 13 columns
anomaly_timeline: 8,423 rows, 13 columns
root_cause_summary: 33 rows, 7 columns
grafana_health: 30,240 rows, 12 columns


In [9]:
# Schema and null profile
for name, df in datasets.items():
    print(f"\n{name} columns:")
    print(sorted(df.columns.tolist()))

    null_pct = (df.isna().mean().sort_values(ascending=False) * 100).round(2)
    print("Top null percentages (%):")
    print(null_pct.head(10))


device_overview columns:
['anomaly_count', 'avg_latency_ms', 'avg_queue_depth', 'avg_throughput_mb_s', 'avg_total_iops', 'avg_util_pct', 'critical_anomaly_count', 'device', 'dominant_workload_pattern', 'high_anomaly_count', 'p95_latency_ms', 'p99_latency_ms', 'sample_count']
Top null percentages (%):
device                       0.0
sample_count                 0.0
avg_total_iops               0.0
avg_throughput_mb_s          0.0
avg_latency_ms               0.0
p95_latency_ms               0.0
p99_latency_ms               0.0
avg_util_pct                 0.0
avg_queue_depth              0.0
dominant_workload_pattern    0.0
dtype: float64

anomaly_timeline columns:
['anomaly_score', 'aqu_sz', 'avg_latency_ms', 'detector_type', 'device', 'metric_name', 'root_cause_hint', 'saturation_score', 'severity', 'timestamp', 'total_iops', 'util_pct', 'workload_pattern']
Top null percentages (%):
device              0.0
timestamp           0.0
metric_name         0.0
detector_type       0.0
sever

In [10]:
# Device coverage comparison across datasets
device_sets = {}
for name, df in datasets.items():
    if "device" in df.columns:
        device_sets[name] = set(df["device"].dropna().astype(str).unique())
        print(f"{name}: {len(device_sets[name])} unique devices")
    else:
        device_sets[name] = set()
        print(f"{name}: no device column")

all_devices = sorted(set().union(*device_sets.values()))
print(f"\nAll unique devices across marts: {len(all_devices)}")
print("\n".join(all_devices))

ref_name = "device_overview"
if device_sets.get(ref_name):
    print(f"\nMissing vs {ref_name}:")
    ref = device_sets[ref_name]
    for name, s in device_sets.items():
        if name == ref_name:
            continue
        missing = sorted(ref - s)
        print(f"{name}: {len(missing)} missing")
        if missing:
            print("  sample:", ", ".join(missing[:20]))

device_overview: 5 unique devices
anomaly_timeline: 5 unique devices
root_cause_summary: no device column
grafana_health: 5 unique devices

All unique devices across marts: 5
dm-0
nvme0n1
nvme1n1
sda
sdb

Missing vs device_overview:
anomaly_timeline: 0 missing
root_cause_summary: 5 missing
  sample: dm-0, nvme0n1, nvme1n1, sda, sdb
grafana_health: 0 missing


In [11]:
# Anomaly and root cause quality checks
if "severity" in anomaly_timeline.columns:
    print("Anomaly severity distribution:")
    print(anomaly_timeline["severity"].value_counts(dropna=False))

if "root_cause_hint" in anomaly_timeline.columns:
    print("\nTop root_cause_hint values:")
    print(anomaly_timeline["root_cause_hint"].fillna("<null>").value_counts().head(15))

if "anomaly_count" in root_cause_summary.columns:
    cols = [c for c in ["device", "root_cause_hint", "anomaly_count"] if c in root_cause_summary.columns]
    print("\nTop root cause rows by anomaly_count:")
    print(root_cause_summary.sort_values("anomaly_count", ascending=False)[cols].head(20))

Anomaly severity distribution:
severity
medium      7373
critical     726
high         324
Name: count, dtype: int64

Top root_cause_hint values:
root_cause_hint
Anomalous behavior detected                                     3967
Composite saturation signal indicates elevated device stress    2264
Latency anomaly detected without clear saturation signal        1119
IOPS anomaly indicates bursty load increase                      570
Latency degradation likely under write-heavy pressure            201
Latency spike likely driven by saturation and queue buildup      126
High IOPS with small requests suggests random IO pressure        102
Latency spike observed during read-heavy demand                   74
Name: count, dtype: int64

Top root cause rows by anomaly_count:
                                      root_cause_hint  anomaly_count
5                         Anomalous behavior detected           2148
21  Composite saturation signal indicates elevated...           1091
1             

In [12]:
# Health/risk screen for dashboard sanity
candidate_cols = ["device", "avg_latency_ms", "util_pct", "aqu_sz", "saturation_score", "anomaly_flag"]
available_cols = [c for c in candidate_cols if c in grafana_health.columns]

health_screen = grafana_health[available_cols].copy()
for c in ["avg_latency_ms", "util_pct", "aqu_sz", "saturation_score"]:
    if c in health_screen.columns:
        health_screen[c] = pd.to_numeric(health_screen[c], errors="coerce")

sort_col = "saturation_score" if "saturation_score" in health_screen.columns else None
if sort_col:
    print("Top 25 rows by saturation_score:")
    print(health_screen.sort_values(sort_col, ascending=False).head(25))

if "anomaly_flag" in health_screen.columns:
    print("\nAnomaly flag distribution:")
    print(health_screen["anomaly_flag"].value_counts(dropna=False))

Top 25 rows by saturation_score:
        device  avg_latency_ms    util_pct     aqu_sz  saturation_score  \
28411      sdb        7.448726   89.742364  43.141044       3871.579221   
28410      sdb      768.498157   94.642991  35.823917       3390.482637   
18980      sdb       76.152507   92.758398  34.267181       3178.568848   
18952      sdb     1771.694136  100.000000  28.243927       2824.392749   
9049       sdb        9.282024   82.440659  27.007887       2226.548046   
28409      sdb        8.732191   88.212151  24.950468       2200.934452   
19785      sdb       25.478788  100.000000  20.727447       2072.744656   
333       dm-0        7.663210   96.081722  19.503819       1873.960508   
28407      sdb     2384.534404   91.308859  20.309345       1854.423092   
9858       sdb       31.032356   56.410115  31.622267       1783.815687   
18317      sdb       43.019806   98.990517  17.130338       1695.741049   
8517       sdb      460.162305   95.599757  16.730832       1599.46